# Van de Vusse Offset-Free MPC Baseline

Canonical Van de Vusse baseline MPC notebook. This workflow loads the direct-linearization system-identification artifacts, augments the nominal model for offset-free MPC, and closes the loop on the nonlinear Van de Vusse plant.

## User Config

In [ ]:
import os
import pickle
from copy import deepcopy
import importlib

import numpy as np

from systems.vandevusse import notebook_params as vandevusse_notebook_params
from utils.notebook_setup import prepare_vandevusse_notebook_env, print_grouped_notebook_summary

importlib.reload(vandevusse_notebook_params)

NB = vandevusse_notebook_params.get_vandevusse_notebook_defaults("baseline")
RUN_MODE = NB["run_mode"]  # "nominal" | "disturb"
DISTURBANCE_PROFILE = NB["disturbance_profile"]  # "none" | "ca0_blocks"
OBSERVER_UPDATE_MODE = str(NB.get("observer_update_mode", "predictor_corrector_current"))
USE_MANUAL_OBSERVER_POLES = bool(NB.get("use_manual_observer_poles", False))
MANUAL_OBSERVER_POLES = np.asarray(
    NB.get("manual_observer_poles", NB["system_setup"]["observer_poles_default"]),
    float,
).copy()
STYLE_PROFILE = NB["style_profile"]
SAVE_PDF = NB["save_pdf"]
VANDEVUSSE_DATA_DIR_OVERRIDE = NB["data_dir_override"]
VANDEVUSSE_RESULTS_DIR_OVERRIDE = NB["results_dir_override"]
RESULT_PREFIX_OVERRIDE = NB["result_prefix_override"]
BASELINE_SAVE_PATH_OVERRIDE = NB["baseline_save_path_override"]

N_TESTS_OVERRIDE = NB["n_tests_override"]
SET_POINTS_LEN_OVERRIDE = NB["set_points_len_override"]
WARM_START_OVERRIDE = NB["warm_start_override"]
TEST_CYCLE_OVERRIDE = NB["test_cycle_override"]
PLOT_START_EPISODE_OVERRIDE = NB["plot_start_episode_override"]

POLE_SEARCH_CFG = deepcopy(NB.get("pole_search", {}))
RUN_POLE_SEARCH = bool(POLE_SEARCH_CFG.get("run_search", False))
POLE_SEARCH_N_SAMPLES = int(POLE_SEARCH_CFG.get("n_samples", 50))
POLE_SEARCH_SEED = int(POLE_SEARCH_CFG.get("seed", 42))
POLE_SEARCH_LOW = float(POLE_SEARCH_CFG.get("low", 0.55))
POLE_SEARCH_HIGH = float(POLE_SEARCH_CFG.get("high", 0.92))
POLE_SEARCH_MODE = str(POLE_SEARCH_CFG.get("mode", "uniform"))  # "uniform" | "mixed"
POLE_SEARCH_TOP_K = int(POLE_SEARCH_CFG.get("top_k", 5))
POLE_SEARCH_N_TESTS_OVERRIDE = int(POLE_SEARCH_CFG.get("n_tests_override", 1))
POLE_SEARCH_SET_POINTS_LEN_OVERRIDE = int(POLE_SEARCH_CFG.get("set_points_len_override", 25))
POLE_SEARCH_TEST_CYCLE_OVERRIDE = list(POLE_SEARCH_CFG.get("test_cycle_override", [False]))

REPO_ROOT, DATA_DIR, RESULT_DIR = prepare_vandevusse_notebook_env(
    data_dir_override=VANDEVUSSE_DATA_DIR_OVERRIDE,
    results_dir_override=VANDEVUSSE_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)


## Imports

In [ ]:
from copy import deepcopy
import importlib

from systems.vandevusse import baseline_mpc as vandevusse_baseline_mpc
from systems.vandevusse import labels as vandevusse_labels
from systems.vandevusse import pole_search as vandevusse_pole_search
from utils.plotting import plot_baseline_mpc_results

importlib.reload(vandevusse_labels)
importlib.reload(vandevusse_baseline_mpc)
importlib.reload(vandevusse_pole_search)

VANDEVUSSE_SYSTEM_METADATA = vandevusse_labels.VANDEVUSSE_SYSTEM_METADATA
prepare_vandevusse_offset_free_mpc_runtime = vandevusse_baseline_mpc.prepare_vandevusse_offset_free_mpc_runtime
run_vandevusse_offset_free_mpc = vandevusse_baseline_mpc.run_vandevusse_offset_free_mpc
run_vandevusse_observer_pole_search = vandevusse_pole_search.run_vandevusse_observer_pole_search
score_vandevusse_observer_poles = vandevusse_pole_search.score_vandevusse_observer_poles

np.set_printoptions(precision=6, suppress=True, linewidth=140)


## Observer Pole Search

In [ ]:
SEARCH_BASELINE_CFG = deepcopy(NB)
SEARCH_BASELINE_CFG["run_mode"] = "nominal"
SEARCH_BASELINE_CFG["disturbance_profile"] = "none"
SEARCH_BASELINE_CFG["observer_update_mode"] = OBSERVER_UPDATE_MODE

default_pole_score = score_vandevusse_observer_poles(
    repo_root=REPO_ROOT,
    baseline_cfg=SEARCH_BASELINE_CFG,
    poles=np.asarray(SEARCH_BASELINE_CFG["system_setup"]["observer_poles_default"], float),
    run_mode="nominal",
    disturbance_profile="none",
    n_tests_override=POLE_SEARCH_N_TESTS_OVERRIDE,
    set_points_len_override=POLE_SEARCH_SET_POINTS_LEN_OVERRIDE,
    test_cycle_override=POLE_SEARCH_TEST_CYCLE_OVERRIDE,
)

pole_search = None
top_pole_candidates = None
BEST_POLES = None
if RUN_POLE_SEARCH:
    pole_search = run_vandevusse_observer_pole_search(
        repo_root=REPO_ROOT,
        baseline_cfg=SEARCH_BASELINE_CFG,
        n_samples=POLE_SEARCH_N_SAMPLES,
        seed=POLE_SEARCH_SEED,
        low=POLE_SEARCH_LOW,
        high=POLE_SEARCH_HIGH,
        mode=POLE_SEARCH_MODE,
        top_k=POLE_SEARCH_TOP_K,
        n_tests_override=POLE_SEARCH_N_TESTS_OVERRIDE,
        set_points_len_override=POLE_SEARCH_SET_POINTS_LEN_OVERRIDE,
        test_cycle_override=POLE_SEARCH_TEST_CYCLE_OVERRIDE,
    )
    top_pole_candidates = pole_search["top_df"][[
        "rank",
        "score",
        "used_poles",
        "observer_spectral_radius",
        "mean_abs_cb_error",
        "mean_abs_T_error",
        "mean_abs_diff_F",
        "mean_abs_diff_QK",
        "mean_abs_diff_cb",
        "mean_abs_diff_T",
    ]].copy()
    if pole_search["best_row"] is not None:
        BEST_POLES = np.asarray(pole_search["best_row"]["used_poles"], float)
        print(f"BEST_POLES = np.array({BEST_POLES.tolist()})")
        print("Default score:", default_pole_score["score"])
        print("Best search score:", pole_search["best_row"]["score"])
    top_pole_candidates
else:
    print("RUN_POLE_SEARCH is False. Set it to True to run the nominal observer-pole random search.")
    print(f"Default score with config poles: {default_pole_score['score']}")


## Runtime Preparation

In [ ]:
BASELINE_CFG = deepcopy(NB)
BASELINE_CFG["run_mode"] = RUN_MODE
BASELINE_CFG["disturbance_profile"] = DISTURBANCE_PROFILE
BASELINE_CFG["observer_update_mode"] = OBSERVER_UPDATE_MODE
if USE_MANUAL_OBSERVER_POLES:
    BASELINE_CFG["observer_poles_override"] = np.asarray(MANUAL_OBSERVER_POLES, float).copy()
BASELINE_CFG["style_profile"] = STYLE_PROFILE
BASELINE_CFG["save_pdf"] = SAVE_PDF
BASELINE_CFG["data_dir_override"] = VANDEVUSSE_DATA_DIR_OVERRIDE
BASELINE_CFG["results_dir_override"] = VANDEVUSSE_RESULTS_DIR_OVERRIDE
BASELINE_CFG["result_prefix_override"] = RESULT_PREFIX_OVERRIDE
BASELINE_CFG["baseline_save_path_override"] = BASELINE_SAVE_PATH_OVERRIDE
BASELINE_CFG["n_tests_override"] = N_TESTS_OVERRIDE
BASELINE_CFG["set_points_len_override"] = SET_POINTS_LEN_OVERRIDE
BASELINE_CFG["warm_start_override"] = WARM_START_OVERRIDE
BASELINE_CFG["test_cycle_override"] = TEST_CYCLE_OVERRIDE
BASELINE_CFG["plot_start_episode_override"] = PLOT_START_EPISODE_OVERRIDE

prepared = prepare_vandevusse_offset_free_mpc_runtime(
    repo_root=REPO_ROOT,
    baseline_cfg=BASELINE_CFG,
)
prepared["system_data"]["A_aug"].shape, prepared["system_data"]["B_aug"].shape, prepared["system_data"]["C_aug"].shape


## Resolved Summary

In [ ]:
print_grouped_notebook_summary(
    "Van de Vusse baseline MPC summary",
    {
        "Paths": {
            "Repo root": REPO_ROOT,
            "Data dir": prepared["data_dir"],
            "Results dir": prepared["result_dir"],
            "Baseline save path": prepared["baseline_save_path"],
        },
        "Run setup": {
            "Run mode": prepared["run_mode"],
            "Disturbance profile": prepared["disturbance_profile_name"],
            "n_tests": prepared["mpc_cfg"]["n_tests"],
            "set_points_len": prepared["mpc_cfg"]["set_points_len"],
            "warm_start": prepared["mpc_cfg"]["warm_start"],
            "test_cycle": prepared["mpc_cfg"]["test_cycle"],
            "plot_start_episode": prepared["plot_start_episode"],
        },
        "System / controller": {
            "delta_t_hours": BASELINE_CFG["system_setup"]["delta_t_hours"],
            "predict_h": prepared["controller_params"]["predict_h"],
            "cont_h": prepared["controller_params"]["cont_h"],
            "Q_out": prepared["controller_params"]["Q_out"].tolist(),
            "R_in": prepared["controller_params"]["R_in"].tolist(),
            "setpoints_phys": prepared["y_sp_scenario_phys"].tolist(),
        },
        "Observer": {
            "update_mode": prepared["observer_update_mode"],
            "manual_override_enabled": USE_MANUAL_OBSERVER_POLES,
            "manual_poles": None if not USE_MANUAL_OBSERVER_POLES else np.asarray(MANUAL_OBSERVER_POLES, float).tolist(),
            "requested_poles": prepared["observer_info"]["poles_requested"].tolist(),
            "used_poles": prepared["observer_info"]["poles_used"].tolist(),
            "used_fallback": prepared["observer_info"]["used_fallback"],
            "spectral_radius": prepared["observer_info"]["observer_error_spectral_radius"],
        },
        "Pole search": {
            "run_search": RUN_POLE_SEARCH,
            "n_samples": POLE_SEARCH_N_SAMPLES,
            "seed": POLE_SEARCH_SEED,
            "search_mode": POLE_SEARCH_MODE,
            "bounds": [POLE_SEARCH_LOW, POLE_SEARCH_HIGH],
            "top_k": POLE_SEARCH_TOP_K,
            "quick_search_n_tests": POLE_SEARCH_N_TESTS_OVERRIDE,
            "quick_search_set_points_len": POLE_SEARCH_SET_POINTS_LEN_OVERRIDE,
        },
        "Disturbance": {
            "c_A0 blocks": BASELINE_CFG["system_setup"]["disturbance_block_values"]["c_A0"].tolist(),
            "T_in blocks": BASELINE_CFG["system_setup"]["disturbance_block_values"]["T_in"].tolist(),
            "thermal-study setpoints": BASELINE_CFG["system_setup"]["thermal_study_setpoints_phys"].tolist(),
        },
        "Reward": prepared["reward_params"],
        "Plotting / export": {
            "style_profile": STYLE_PROFILE,
            "save_pdf": SAVE_PDF,
            "result_prefix": prepared["result_prefix"],
        },
    },
)


## Run Baseline MPC

In [ ]:
result_bundle = run_vandevusse_offset_free_mpc(prepared)
with open(prepared["baseline_save_path"], "wb") as handle:
    pickle.dump(result_bundle, handle)
print(f"Saved canonical baseline bundle to: {prepared['baseline_save_path']}")


## Plotting And Export

In [ ]:
out_dir = plot_baseline_mpc_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": prepared["result_dir"],
        "prefix_name": prepared["result_prefix"],
        "start_episode": prepared["plot_start_episode"],
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
    },
)

print(f"Baseline plot/output directory: {out_dir}")
print(f"Baseline pickle: {prepared['baseline_save_path']}")
print(f"Observer update mode used: {result_bundle['observer_update_mode']}")
print("Observer poles used:", np.asarray(result_bundle["observer_poles_used"], float))
print("Input saturation summary:", result_bundle.get("input_saturation_summary"))
